卷积神经网络（convolutional neural networks，CNN）是机器学习利用自然图像中一些已知结构的创造性方法。

平移不变性（translation invariance）：不管检测对象出现在图像中的哪个位置，神经网络的前面几层应该对相同的图像区域具有相似的反应，即为“平移不变性”。

局部性（locality）：神经网络的前面几层应该只探索输入图像中的局部区域，而不过度在意图像中相隔较远区域的关系，这就是“局部性”原则。最终，可以聚合这些局部特征，以在整个图像级别进行预测。

互相关运算，卷积层是个错误的叫法，因为它所表达的运算其实是互相关运算（cross-correlation），而不是卷积运算。

在二维互相关运算中，卷积窗口从输入张量的左上角开始，从左到右、从上到下滑动。 当卷积窗口滑动到新一个位置时，
包含在该窗口中的部分张量与卷积核张量进行按元素相乘，得到的张量再求和得到一个单一的标量值，由此我们得出了这一位置的输出张量值



In [1]:
import torch
from torch import nn
def corr2d(x,k):
    h,w=k.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y


In [2]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

卷积层



In [ ]:
import torch
class Conv2D(nn.module):
    def __init__(self,kernel_size):
        super.__init__()
        # 初始化参数
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1)) 
    def forward(self,x):
        return corr2d(x,self.weight)+self.bias

卷积层使用案例

torch.nn.Conv2d类是创建卷积层的类

conv_layer = nn.Conv2d(
    in_channels,      # 输入通道数（例如RGB图像为3）
    out_channels,     # 输出通道数（卷积核数量，即特征图数量）
    kernel_size,      # 卷积核大小（整数或元组，如3 或 (3,3)）
    stride=1,         # 步长
    padding=0,        # 填充
    dilation=1,       # 空洞卷积
    groups=1,         # 分组卷积
    bias=True         # 是否添加偏置
)

In [ ]:
# 参考6.2.3. 图像中目标的边缘检测


import torch
# 构造一个二维卷积层，它具有1个输出通道和形状为（1，2）的卷积核
conv2d = torch.nn.Conv2d(1,1, kernel_size=(1, 2), bias=False)

# 这个二维卷积层使用四维输入和输出格式（批量大小、通道、高度、宽度），
# 其中批量大小和通道数都为1
X = torch.ones((6, 8))
X[:, 2:6] = 0

X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))



In [6]:
lr = 3e-2  # 学习率

for i in range(10):
    Y_hat = conv2d(X.float())
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    # 迭代卷积核
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 191524931006.709
epoch 4, loss 37642507852236062241849344.000
epoch 6, loss 7398295486384927321232845797819242512384.000
epoch 8, loss 1454068068072250446706063724720311919061687008747323392.000
epoch 10, loss 285783985163842804865229616534259101923808572228544715096091615297536.000


特征映射和感受野是什么

填充和步幅

填充padding，可以控制输出层shape
步幅stride， 控制卷积核移动距离

conv2d = nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
卷积核宽度是5，高度是3，宽方向上填充一行，在高方向上移动跨度是3，宽方向上移动跨度是4

In [ ]:
# 我们创建一个高度和宽度为3的二维卷积层，并在所有侧边填充1个像素。
# 给定高度和宽度为8的输入，则输出的高度和宽度也是8。

In [ ]:
import torch
# padding=1 就是在输入张量的 上下左右各补一行/列0
conv2d = torch.nn.Conv2d(1, 1, kernel_size=3, padding=1)
print(conv2d(torch.rand([1,1,8,8])).shape)  # torch.Size([1, 1, 8, 8])

conv2d_no_padding=torch.nn.Conv2d(1, 1, kernel_size=3, padding=0)
print(conv2d_no_padding(torch.rand([1,1,8,8])).shape) # torch.Size([1, 1, 6, 6])

torch.Size([1, 1, 8, 8])
torch.Size([1, 1, 6, 6])


步幅也会降低输出大小



画一个 10×10 的格子，最外圈标0
内圈8×8随便填字母或数字
拿一张小纸片剪一个 3×3 的方孔​ 作为卷积核
从左上角开始，把小纸片放在格子上
每次向右移动 2格，到头后换行再向左移回来，同样每次向下移动 2格
数一数一共放了多少次 —— 正好 16次，对应4×4的输出

In [ ]:
import torch
conv2d = torch.nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)
print(conv2d(torch.rand([1,1,8,8])).shape)  # torch.Size([1, 1, 4, 4])


torch.Size([1, 1, 4, 4])


多输入多通道

In [14]:
import torch

X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])
print(X.shape,"\n",K.shape)


torch.Size([2, 3, 3]) 
 torch.Size([2, 2, 2])


AttributeError: module 'torch.nn' has no attribute 'corr2d'

汇聚层